# Sonido en Python

#### Ya sabemos generar sonidos en Python!

#### Sonido digital = secuencia de números (muestras o samples)


<br><br>


<center>
<img src="media/noise-code.png" width=80% />
</center>

<br><br>



<center>
<img src="media/sin-code.png" width=80% />
</center>



## Solo nos falta que suenen...

<center>
<img src="media/sounddevice.png" width=80% />
</center>





### Nos interesará:



- <font color='darkgreen'>Reproducir</font>: enviar las muestras a la tarjeta de sonido (DAC)

- <font color='darkgreen'>Generar</font>: producir muestras mediante algoritmos $\leadsto$ síntesis digital de sonido


- <font color='darkgreen'>Procesar</font>: transformar las muestras mediante algoritmos (DSP, mezcla)

- <font color='darkgreen'>Grabar</font>: recoger el sonido
    (**muestrear**, ADC) con una tarjeta de sonido y almacenarlo en formato digital



##### Para ello utilizaremos las librerías: 

- <font color='darkgreen'>**numpy**</font>: arrays eficientes no utilizar listas Python para las muestras!
  
  - <font color='darkgreen'>**scipy**</font>: para algoritmos específicos de procesamiento de audio

- <font color='darkgreen'>**sounddevice**</font>: mapping (bindings) en Python de la librería **PortAudio**
  (entrada/salida de audio multiplataforma)

- <font color='darkgreen'>**soundfile**</font>: carga y guardado de archivos de sonido


In [ ]:
# instalamos/actualizados los paquetes necesarios
import sys

!pip install sounddevice soundfile --upgrade
!pip install numpy --upgrade
!pip install ipywidgets --upgrade



   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ---------------------------------- ----- 786.4/914.9 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 2.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 1.7 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 2.7 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------

# Reproductor básico

In [ ]:
import numpy as np         # arrays    
import sounddevice as sd   # modulo de conexión con portAudio
import soundfile as sf     # para lectura/escritura de wavs

# leemos wav: data (array numpy con samples y SRATE)
data, SRATE = sf.read('media/ex1.wav')  

# informacion de wav
print("\n\nInfo del wav " )
print("  Sample rate ", SRATE)  # leído del archivo
print("  Sample format: ", data.dtype)
print("  Num channels: ", len(data.shape))
print("  Len: ", data.shape[0])
  
# bajamos volumen: operación sobre array de numpy
data = data * 0.5

# a reproducir!
sd.play(data, SRATE)

# mantenemos de ejecución activa hasta que termine la reproducción
# sd.wait()
# no se necesita en un notebook!! 



Info del wav 
  Sample rate  48000
  Sample format:  float64
  Num channels:  2
  Len:  4432286


In [ ]:
sd.stop()  # parar reproducción 

# Buffering: procesamiento por **chunks**

El player anterior lee todos los datos de golpe y los envía al stream,
**bloqueando** el proceso de ejecución $\leadsto$ no se pueden manipular las muestras durante la reproducción

- Ej: no se puede modificar el volumen una vez arrancada la reproducción, ni aplicar ningún tipo de efecto.



## Solución: procesamiento por bloques (chunks)

<center>
<img src="media/chunks.png" width="50%" />
</center>


Los chunks serán arrays de numpy:
- de tamaño prefijado (pequeño, $2^6...2^{12}$) 
- obtenidos mediante *slicing*
- que se pueden *procesa* individualmente 
- y envíar *secuencialmente* (de uno en uno) a sounddevice

### Reproductor con *fade out*: bajamos paulatinamente el volumen

In [4]:
import numpy as np         # arrays    
import sounddevice as sd   # modulo de conexión con portAudio
import soundfile as sf     # para lectura/escritura de wavs

CHUNK = 2048   # tamaño CHUNK o bloque

# Leemos wav. Por defecto lee float64: no soportado por portAudio, 
# Convertimos directamente la conversion a float32
data, SRATE = sf.read('media/ex1.wav',dtype=np.float32)

# stream de salida
stream = sd.OutputStream( # creamos stream 
    samplerate = SRATE,            # frec de muestreo 
    blocksize  = CHUNK,            # tamaño del bloque
    channels   = len(data.shape))  # num de canales

# arrancamos stream
stream.start() 

prog = ['-','\\','|','/'] # para decorar
vol = 0.8      # volumen
numBloque = 0  # contador de bloques
end = False # para detección de últimol chunck 

while not(end): 
    # slice de data (no copia!). Si no hay suficientes samples, extrae los que queden
    bloque = data[numBloque*CHUNK : (numBloque+1)*CHUNK]
    
    if bloque.shape[0]<CHUNK: # ultimo bloque? -> se hace esta vuelta del bucle y terminamos
        end = True 
    
    bloque *= vol # modificamos volumen del bloque en cada vuelta del bucle!
    stream.write(bloque) # escribimos al stream de sounddevice
    

    vol -= 0.01  # bajamos volumen en cada bloque
    if vol<=0:   # terminamos si vol<=0
        end = True  

    numBloque += 1
    print(f'\rProgreso: {prog[numBloque%4]}   bloque: {numBloque}   vol: {vol}', end='')

print(f'\nÚltimo bloque: {bloque.shape[0]} samples')
stream.stop()  # cerramos stream
stream.close() 


Progreso: -   bloque: 80   vol: -5.30825383648903e-16
Último bloque: 2048 samples


# Hebras de ejecución y callBacks

En todos los ejemplos anteriores, la llamada `stream.write(...`

- sigue siendo "algo" <span style='color:darkgreen'>**bloqueante**</span>: bloquea la ejecución hasta que se completa el envío de datos al flujo

   $\leadsto$ tenemos control *a intervalos*: podemos interactuar entre envíos de bloques, p.e., para variar el volumen durante la reproducción

- En general, la versión que tenemos sería suficiente para muchas aplicaciones    

Una opción para tener más control: crear una nueva <font color='darkgreen'>**hebra de ejecución**</font> con la reproducción para no tener ningún bloqueo en la hebra principal

- Aun así... la variación de volumen solo tiene efecto entre CHUNKS, pero esto es asumible (siempre que sean de tamaño relativamente pequeño)


### Sounddevice ya gestiona las hebras (no es necesario crear hebras explícitamente)

In [5]:
# stream de salida con callBack
stream = sd.OutputStream(
  samplerate = SRATE, 
  channels = len(data.shape),
  blocksize = CHUNK, 
  callback = callback) # función callback se llama bajo demanda de chunks

NameError: name 'callback' is not defined

$\leadsto$ se invoca a la función **callback** cuando el stream demanda nuevo audio para reproducir


## La función *callback*. Prototipo

```python
callback(outdata,      # datos de salida
         frames,       # num nBloques a procesar por el stream = len de outdata
         time_info,    # estructura con current_frame_frameTime, etc
         status_flags) 
```             

En la práctica, lo esencial es: 

- **rellenar** `outdata` con los samples de salida (no crear un nuevo vector!)  y

- tener cuidado con el *shape* del array que se copia a `outdata`: espera el formato  *(frames, nChannels)* 
    - En mono, numpy genera arrays con formato *(n,)* y hay que convertir a *(n,1)* con *reshape*:
    
        ```if (len(data.shape)==1): data = np.reshape(data,(data.shape[0],1))```

Menos esencial:

- `frames` el número de frames = long de `outdata` = `CHUNK`

-   `time_info`: contiene `input_buffer_adc_time`,
    `current_frame_time` y `output_buffer_dac_time` (ver documentación de PortAudio)
-   `status_flags` (ver documentación de PortAudio)



# Reproductor con callback


In [ ]:
import numpy as np         
import sounddevice as sd   
import soundfile as sf     

CHUNK = 2048
data, SRATE = sf.read('media/ex1.wav',dtype=np.float32)

# para archivos mono, devuelve un array de la forma data.shape = (n,)
# para rellenar el array outdata del callback se necesita hacer explícito el número de canales
# convertir el data.shape  (n,) -> (n,1) 
if (len(data.shape)==1): 
    data = np.reshape(data,(data.shape[0],1))
    # otra forma data = data.reshape(-1, 1)

# info del wav
print(f"SRATE: {SRATE}   Format: {data.dtype}   Channels: {len(data.shape)}    Len: {data.shape[0]}")


# contador de frames, global
current_frame = 0
def callback(outdata, frames, time, status):
    global current_frame      # para actualizarlo en cada callBack
    if status: print(status)

    # ojo, este print es muy lento... puede provocar underruns
    # solo para depuración en ete ejemplo
    print(f"\rNum Bloque: {current_frame//CHUNK}  frame: {current_frame}", end='') 

    # escribimos los samples correspondientes en el outdata que viene dado
    bloque = data[current_frame : current_frame+CHUNK]

    # tamaño del blque leido
    chunksize = bloque.shape[0]

    outdata[:chunksize] = bloque
    # es una forma EFICIENTE de rellenar outdata, copiando los samples del bloque 
    # Es similar a 
    #   for i in range(chunksize): outdata[i] = bloque[i]
    # pero este es for es demasiado lento (es un for de python)-> underruns!!

    # NO funcionaría hacer outdata = data[current_frame:current_frame + chunksize]
    # compartiría referencias (objetos array de numpy)
    # outdata viene dado y hay que rellenar su contenido

    
    if chunksize < frames: # ha terminado?
        print('fin')
        outdata[chunksize:] = 0 # rellenamos con 0's el resto de outdata
        raise sd.CallbackStop()

    # actualizamos frame con los frames procesados    
    current_frame += chunksize


# stream de salida con callBack
stream = sd.OutputStream(samplerate=SRATE, 
                         channels=len(data.shape),
                         callback=callback, 
                         blocksize=CHUNK)
stream.start()

# este while solo sirve para mantener la hebrea principal activa y mostrar el progreso de los bloques en el notebook
# pero no es necesario para la reproducción en sí en el notebook
while stream.active:
    pass
    

SRATE: 44100   Format: float32   Channels: 2    Len: 4109807
Num Bloque: 771  frame: 1579008

KeyboardInterrupt: 

Num Bloque: 954  frame: 1953792

## ... como paramos??

In [7]:
# parada y liberación de recursos (no se puede ejecutar si se ha puesto el while arriba!!)

stream.stop()
stream.close()

## Entendiendo el array `outdata`?

Hay que copiar `CHUNK` muestras de `data` en`outdata`, rellenando el `outdata` que viene dado, sin generar un nuevo array

### mini ejemplo para ver la copia de muestras

In [9]:
import numpy as np
CHUNK = 5  # tamaño del chunk

# next es un # "iterador" que va dando sucesivos chunks
# en cada llamada
current_frame = 0
def next(outdata): 
    global current_frame 

    # nuevo bloque de tamaño CHUNK (si queda suficiente)
    bloque = data[current_frame:current_frame+CHUNK]

    # tamaño del bloque obtenido (puede estar incompleto)
    size = bloque.shape[0]

    # y lo ponemos en outdata, al principio
    outdata[:size] = data[current_frame:current_frame+size]
    current_frame += size

    # si out no está completo, rellenamos con 0s
    if size<CHUNK:
        outdata[size:] = 0
        print('FIN')

# data con 8 eltos [0..7]
data = np.arange(8,dtype=np.float32)

# outdata de tamaño CHUNK (5), con ceros
outdata = np.zeros(CHUNK)

print('Inicialmente (antes de llamar a next)')
print('data:',data)
print('outdata: ',outdata)


Inicialmente (antes de llamar a next)
data: [0. 1. 2. 3. 4. 5. 6. 7.]
outdata:  [0. 0. 0. 0. 0.]


In [11]:

# ahora ejecutar sucesivas veces
# outdata es siempre el mismo array que se va rellenando con sucesivos slices (copias)

next(outdata)
print(outdata)


FIN
[5. 6. 7. 0. 0.]


# Interacción en tiempo real... nuestro primer efecto en tiempo real!

### Inciso: cómo modificar el valor de una variable (volumen) mientras suena la canción?

- Podemos leer input de teclado 

- ... pero la lectura de teclado *sencilla* es **bloqueante**: hay que pulsar intro para recoger el input

In [ ]:
x = input("Nombre: ") # bloquea ejecución hasta pulsación de intro
print(f"Te llamas {x}")

## Ipywidgets: botones, sliders, etc 

- Muy cómodo para trabajar en los notebooks

- Ojo: esto no es útil fuera de los notebooks. No es esencial para el tratamiento de audio que hacemos

- Más adelante generalizaremos con la librería TKInter 

In [ ]:
import numpy as np         
import sounddevice as sd   
import soundfile as sf     
from ipywidgets import interact


CHUNK = 2048
data, SRATE = sf.read('media/ex1.wav',dtype=np.float32)

if (len(data.shape)==1): data = np.reshape(data,(data.shape[0],1))
print(f"SRATE: {SRATE}   Format: {data.dtype}   Channels: {len(data.shape)}    Len: {data.shape[0]}")

# volumen incial
vol = 0.5

# contador de frames, global
current_frame = 0
def callback(outdata, frames, time, status):
    global current_frame       # para actualizarlo en cada callBack
    if status: print(status)

    # ojo, este print es muy lento... puede provocar underruns
    #print(f"\rNum Bloque: {current_frame//CHUNK}  frame: {current_frame}", end='') 

    # escribimos los samples correspondientes en el outdata que viene dado
    bloque = data[current_frame : current_frame+CHUNK] * vol
        
    chunksize = bloque.shape[0]

    outdata[:chunksize] = bloque       
    
    if chunksize < frames: # ha terminado?
        print('fin')
        outdata[chunksize:] = 0 # rellenamos con 0's el resto de outdata
        raise sd.CallbackStop()

    # actualizamos frame con los frames procesados    
    current_frame += chunksize


# stream de salida con callBack
stream = sd.OutputStream(samplerate=SRATE, channels=len(data.shape),
    callback=callback, blocksize=CHUNK)
stream.start()


# controles de reproducción: volumen y parada

def volCtrl(v):
    global vol
    vol = v
    #print(f"\rvol: {vol}  frame: {current_frame}", end='') 
#                   (min, max, step)    
interact(volCtrl, v=(0.0,1.0,0.01))


def pauseCtrl(pause):
    if pause==True:
        stream.stop()        
    else:
        stream.start()   
# lo interpreta como un checkbox
interact(pauseCtrl, pause=False)       



def stopCtrl(stop):
    if stop==True:
        stream.stop()
        stream.close()
interact(stopCtrl, stop=False)        


SRATE: 44100   Format: float32   Channels: 2    Len: 4109807


# Grabación con callBack (incluso más fácil)

In [ ]:
import numpy as np         
import sounddevice as sd   
import soundfile as sf     
from ipywidgets import interact


SRATE = 48000
CHUNK = 1024

# buffer para acumular grabación.
# (0,1): de tamaño 0 (vacío), y con 1 canal 
buffer = np.empty((0, 1), dtype=np.float32)

def callback(indata, frames, time, status):
    global buffer
    # concatenamos indata al buffer
    buffer = np.append(buffer,indata)

# stream de entrada con callBack
stream = sd.InputStream(
    samplerate=SRATE, dtype=np.float32,  channels=1,
    blocksize=CHUNK, callback=callback)

# arrancamos stream -> arrancamos grabación
stream.start()


# control de parada de grabación
def recCtrl(stopRec):
    if stopRec==True:
        stream.stop()        
        stream.close()
        sf.write('grabacion.wav', buffer, SRATE)

interact(recCtrl, stopRec=False)   



interactive(children=(Checkbox(value=False, description='stopRec'), Output()), _dom_classes=('widget-interact'…

<function __main__.recCtrl(stopRec)>

In [13]:

# reproducción de la grabación
data, SRATE = sf.read('grabacion.wav')  
sd.play(data)
#sd.wait()
